# PARTIE A — Chargement & choix des variables

Partie 1 Charger le fichier excel et le convertir en cvs

In [ ]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel('../data/dataset_assurance_ML.xlsx')
df.to_csv('data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig')

df =  pd.read_csv('../data/dataset_assurance_ML.csv')
if df.empty:
    print ("Date set vide ")
else:
    print("Donnée bien chargées")

NameError: name 'pd' is not defined

In [7]:
df.shape
df.head()

,N° Police,Nom,Prénom,Sexe,Âge,Catégorie Prof.,Salaire Annuel (€),Ville,Code Postal,Type Contrat,...,Type Véhicule,Usage Véhicule,Puissance Fiscale (CV),Valeur Véhicule (€),Coeff. Bonus-Malus,Nb Sinistres (3 ans),Montant Sinistres (€),Dernier Sinistre,Score Risque (0-100),Résiliation
0,POL-2026-00001,Lambert,Fatou,F,32,Cadre,46800,Paris,75015,Bronze,...,Berline,Privé,8,21300,0.96,2,10017,1-2 ans,38,0
1,POL-2026-00002,Lefebvre,Samuel,H,54,Employé,28500,Paris,75018,Gold,...,Citadine,Trajet travail,5,14900,0.93,2,16071,2-3 ans,51,1
2,POL-2026-00003,Simon,Léa,F,42,Indépendant,32200,Toulouse,31004,Platine,...,SUV,Trajet travail,7,46500,0.75,1,1964,1-2 ans,8,0
3,POL-2026-00004,David,Amina,F,38,Employé,30700,Paris,75001,Bronze,...,SUV,Trajet travail,9,33400,0.50,0,0,Aucun,0,0
4,POL-2026-00005,Martinez,Michel,H,41,Employé,29600,Paris,75011,Platine,...,Sportive,Professionnel,17,38500,1.00,0,0,Aucun,43,0


In [25]:
TARGET = 'Résiliation'
print(df[TARGET].value_counts())

print ('en ourcentage') 
pourcentages = df[TARGET].value_counts(normalize=True) * 100
pourcentages.round(2)

Résiliation
0    442
1     58
Name: count, dtype: int64
en ourcentage


Résiliation
0    88.4
1    11.6
Name: proportion, dtype: float64

In [26]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET]))

Résiliation       0   1
Statut Contrat         
Actif           425   0
Résilié           0  45
Suspendu         17  13


je remarque avec la variable `status contrat`  nous avons un `100%` en actif et `100%` en resilié 
saut le dernier cas suspendu qui pose un proble 

`Nom` nous ne pouvons pas utilser  cette varible car déjà `etiquété` 

In [31]:
num_cols = [
    'Âge', 'Salaire Annuel (€)', 
    'Prime Annuelle (€)', 
    'Ancienneté (mois)',
    'Coeff. Bonus-Malus', 
    'Nb Sinistres (3 ans)',
    'Montant Sinistres (€)', 
    'Score Risque (0-100)'
    ]
cat_cols = ['Type Contrat', 
            'Catégorie Prof.', 
            'Usage Véhicule', 
            'Dernier Sinistre'
            ]

X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape)

(500, 12) (500,)


Etape 5:

En Machine Learning, calculer la corrélation d'une variable numérique avec la variable cible (Résiliation ou TARGET) permet de mesurer l'intensité du lien linéaire entre cette variable et le fait de résilier.
    - \Une corrélation proche de $+1$ : Quand la variable augmente, le risque de résiliation augmente fortement.
    - \Une corrélation proche de $-1$ : Quand la variable augmente, le risque de résiliation diminue fortement.
    - \Une corrélation proche de $0$ : La variable n'a pas de lien linéaire évident avec la résiliation.

In [33]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False))
print("#############################""")
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values())

Score Risque (0-100)     0.228
Prime Annuelle (€)       0.213
Montant Sinistres (€)    0.171
Coeff. Bonus-Malus       0.166
Nb Sinistres (3 ans)     0.165
Ancienneté (mois)       -0.024
Âge                     -0.130
Salaire Annuel (€)      -0.187
dtype: float64
#############################
Dernier Sinistre
Aucun        0.08
< 6 mois     0.10
2-3 ans      0.16
1-2 ans      0.16
6-12 mois    0.24
Name: Résiliation, dtype: float64


les trois variables sont: `'Résiliation', 'Statut Contrat', 'ID_Client'`

In [40]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.20, random_state=42, stratify= y)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2))

(400, 12) (100, 12)
0.12 0.12


In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
preprocessor = ColumnTransformer([
('num', StandardScaler(), num_cols),
('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
candidats = {
'Régression Logistique': LogisticRegression(
max_iter=1000, class_weight='balanced', random_state=42),
'Random Forest': RandomForestClassifier(
n_estimators=300, max_depth=4, min_samples_leaf=10,
class_weight='balanced', random_state=42),
}
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
for nom, algo in candidats.items()}

In [43]:
from sklearn.model_selection import cross_val_score
for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

Régression Logistique  AUC = 0.684 ± 0.133
Random Forest          AUC = 0.696 ± 0.095


Performance globale modérée (AUC $\approx$ 0,69) :Un score AUC de 0,5 correspond à un choix aléatoire (jet de pièce), tandis qu'un score de 1,0 représente un modèle parfait. Avec environ 0,69, vos deux modèles apprennent un signal utile, mais leurs performances restent moyennes.


Avantage pour le Random Forest :Le Random Forest obtient une meilleure capacité de discrimination (AUC légèrement plus élevé à 0,696) et, surtout, une meilleure stabilité ($\pm$ 0,095 contre $\pm$ 0,133 pour la Régression Logistique). Une variance plus faible indique qu'il réagit de façon plus constante d'un pli de validation croisée à un autre.Variabilité importante ($\pm$ 0,095 à $\pm$ 0,133) :L'écart-type est relativement élevé. Cela s'explique généralement par la taille réduite de l'échantillon ou par le déséquilibre de la variable cible (environ 10 % de résiliations).

In [46]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
confusion_matrix, classification_report)
pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
print('Accuracy :', round(accuracy_score(y_test, y_pred),3))
print('F1:', round(f1_score(y_test, y_pred),3))
print('ROC-AUC :', round(roc_auc_score(y_test, y_proba),3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

Accuracy : 0.81
F1: 0.387
ROC-AUC : 0.795
[[75 13]
 [ 6  6]]
              precision    recall  f1-score   support

       Reste       0.93      0.85      0.89        88
     Résilie       0.32      0.50      0.39        12

    accuracy                           0.81       100
   macro avg       0.62      0.68      0.64       100
weighted avg       0.85      0.81      0.83       100



Etape 12 sauvegarde du pipe ligne

In [50]:
import joblib, os
joblib.dump(pipeline, '../models/pipeline_resiliation.pkl')
print(os.path.getsize('../models/pipeline_resiliation.pkl') / 1024, 'Ko')

529.595703125 Ko


Etape 13 Sauvegarde des eto Donée 

In [52]:
import json
meta = {
'modele': 'Random Forest',
'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
'num_cols': num_cols,
'cat_cols': cat_cols,
'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),
'median': float(X[c].median())} for c in num_cols},
'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
}
with open('../models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [53]:
modele = joblib.load('../models/pipeline_resiliation.pkl')
client = pd.DataFrame([{
'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3,
'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72,
'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur',
'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol',
}])
print('Classe :', modele.predict(client))
print('Proba :', modele.predict_proba(client)[0, 1].round(3))

Classe : [1]
Proba : 0.594


Etape 15 erreur classique 

In [54]:
try:
    modele.predict(client.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)

ERREUR : columns are missing: {'Score Risque (0-100)'}
